# Worked Example: Artifact Pass on Bipolar Continuous Data

## Goal
Run misc + IED detectors on `sample_ieeg_bp.fif` and plot flagged time ranges.


In [ ]:
from pathlib import Path
from LFPAnalysis import build_basic_pipeline_config, run_pipeline

config = build_basic_pipeline_config(
    Path('../../data/sample_ieeg_bp.fif'),
    file_format='mne',
    artifact_methods=['misc', 'ied'],
    preload=True,
)
result = run_pipeline(config)
misc_table = result.artifact_tables['misc']
ied_table = result.artifact_tables['ied']
print(f'Misc events: {len(misc_table)}, IED events: {len(ied_table)}')
print(misc_table.head())

## Plot flagged time ranges on a short raw segment

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

raw = result.referenced if result.referenced is not None else result.raw
chan = 'racas1-racas2' if 'racas1-racas2' in raw.ch_names else raw.ch_names[0]
sfreq = float(raw.info['sfreq'])
t0, t1 = 240.0, 250.0
start, stop = int(t0 * sfreq), int(t1 * sfreq)
segment = raw.get_data(picks=[chan])[0, start:stop]
times = np.arange(len(segment)) / sfreq + t0
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(times, segment, 'k', lw=0.7)
chan_events = misc_table[misc_table['channel'] == chan] if len(misc_table) else misc_table
for _, row in chan_events.iterrows():
    t = float(row['time_seconds'])
    if t0 <= t <= t1:
        ax.axvspan(t - 0.05, t + 0.05, color='C1', alpha=0.35)
ax.set(xlabel='Time (s)', ylabel='Amplitude', title=f'{chan} with misc flags')
fig.tight_layout()
plt.show()

## Next step

Chapter 06 (`06_first_baseline`) covers baseline correction on epochs.